# AITR Lab 02 — Full experiment on Colab (protocol v4)

Runs everything in one go: translate the 500 test questions to Hindi and Odia (and back), score Qwen2.5-1.5B on 5 conditions, evaluate, and draw the figures. Expected time **~30–40 min**.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then click **Runtime → Run all**. Only two prompts need you: Drive access (click Allow) and the Hugging Face token box.

**Never paste your token into a cell**, only into the login box.

## 1. GPU check (must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Google Drive + unpack project (upload `AITR_upload.zip` to My Drive first)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/AITR_upload.zip -d /content/
%cd /content/AITR_Lab02_MAI2026015_MultiSciQA

## 3. Libraries (transformers 4.51.3: newer versions crash IndicTrans2)

In [ ]:
!pip install -q 'transformers==4.51.3' 'IndicTransToolkit==1.1.1' sentencepiece sacrebleu pyyaml pytest
import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 4. Hugging Face login

In [ ]:
from huggingface_hub import login
login()

## 5. Pre-test checklist: frozen protocol hash, leakage test on all 1,000 items, metric tests

In [ ]:
!python -m pytest -q 2>&1 | tail -3

## 6. Translate test set: Hindi, then Odia (EN→X and X→EN)

In [ ]:
!python scripts/01_translate.py --lang hi --split test 2>&1 | grep -v -i warn | tail -22

In [ ]:
!python scripts/01_translate.py --lang or --split test 2>&1 | grep -v -i warn | tail -22

## 7. Score Qwen2.5-1.5B on the TEST set (5 conditions)

In [ ]:
!python scripts/02_score.py --model Qwen/Qwen2.5-1.5B-Instruct --condition EN --split test --confirm-test 2>&1 | grep -E '"(n|seconds|accuracy|macro_f1)"'
for lang in ['hi', 'or']:
    for cond in ['NATIVE', 'TT']:
        print('==', cond, lang)
        !python scripts/02_score.py --model Qwen/Qwen2.5-1.5B-Instruct --condition $cond --lang $lang --split test --confirm-test 2>&1 | grep -E '"(n|seconds|accuracy|macro_f1)"'

## 8. Evaluate (tables + statistics) and draw figures

In [ ]:
!python scripts/03_evaluate.py --split test
!python scripts/04_figures.py --split test

## 9. Save everything to Drive

In [ ]:
!pip freeze | grep -i -E '^(torch|transformers|indictranstoolkit|sentencepiece|tokenizers|sacrebleu|scipy|numpy)==' > results/colab_env.txt
!nvidia-smi --query-gpu=name,driver_version --format=csv >> results/colab_env.txt
!zip -q -r /content/drive/MyDrive/results_out.zip data/translations results
print('Saved to My Drive: results_out.zip')

## 10. Done
Download **results_out.zip** from Google Drive into your Mac's **Downloads** folder and tell Claude.